# Imports

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer
from transformers import DataCollatorForMultipleChoice

: 

# Set Up Variables

In [ ]:
MODEL = "/kaggle/input/datasets/kalyandath/deberta-pretrained/"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

# Set up Model and Tokenizer

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMultipleChoice.from_pretrained(MODEL)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

# Tokenization

In [ ]:
def preprocess(data):
    output = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    label_map = {'A':0, 'B':1,'C':2, 'D':3, 'E':4}

    prompts = []
    options = []
    answer = []

    for i in range(len(data["prompt"])):
        prompts += [data["prompt"][i]] * 5
        options += [data[x][i] for x in label_map.keys()]
        answer.append(label_map[data["answer"][i]])

    tokenized_input = tokenizer(prompts, options, truncation = True, max_length = 256)
    
    for i in range(0, len(tokenized_input["input_ids"]),5):
        output["input_ids"].append(tokenized_input["input_ids"][i:i+5])
        output["attention_mask"].append(tokenized_input["attention_mask"][i:i+5])
        output["labels"].append(answer[i//5])

    print(output["input_ids"][0])
    return output

# Load the Test CSV

In [ ]:
## Load the test csv
test_df = pd.read_csv(TEST_PATH)
test_df["answer"] = "A"
test_dataset = Dataset.from_pandas(test_df)

# Apply preprocess function to get tokenized data for test.csv
tokenized_test = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[[1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 262, 454, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 1026, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 467, 5465, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 26

# Run the Predictions

In [ ]:

data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
inference_trainer = Trainer(
    model = model,
    data_collator = data_collator
)
print("Running Predictions....")
predictions = inference_trainer.predict(tokenized_test)
logits = predictions.predictions

Running Predictions


# Prepare Submission Data

In [25]:
top_3_indices = np.argsort(logits, axis=1)[:,-1:-4:-1]
index_to_letter = {0:"A", 1:"B", 2:"C", 3:"D", 4: "E"}
predicted_strings = []

for row in top_3_indices:
    letters = [index_to_letter[idx] for idx in row]
    predicted_strings.append(" ".join(letters))

submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predicted_strings
})

# Submission For Pretrained Model

In [ ]:

submission_df.to_csv("submission.csv", index= False)
print(submission_df.head())

   id prediction
0   1      A D B
1   2      B D A
2   3      B E D
3   4      E C A
4   5      C D A
